# 03 · Evaluation: base vs +continued pre-training vs +instruction tuning
Same held-out data for all three models. (1) **Perplexity** on text none of them saw: PubMed Central articles, MedlinePlus, Ghana STG/EML, and WikiText-2 as a general-English control. (2) **Zero-shot multiple-choice accuracy** on MedQA, MedMCQA, PubMedQA. (3) Side-by-side answers to GP vignettes. Results are saved to Drive as JSON and PNG.

In [ ]:
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install -q -U "transformers>=5"      # Qwen3.5 needs transformers v5; restart the runtime if Colab asks, then skip this cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, zipfile
BASE = '/content/drive/MyDrive/health-llm'            # put colab_health_bundle.zip here
os.makedirs(BASE, exist_ok=True)
if not os.path.exists('cpt_train.jsonl'):
    zipfile.ZipFile(f'{BASE}/colab_health_bundle.zip').extractall('.')
sys.path.insert(0, '.')
MODEL = 'unsloth/Qwen3.5-2B-Base'    # base (pre-trained only) model, the right start for continued pre-training. 4B does not fit a T4: Unsloth forces float32 on Qwen3.5 there (no bf16), ~16 GB of weights vs 15 GB VRAM
LOAD = dict(load_in_4bit=False, load_in_16bit=True, full_finetuning=False)   # Unsloth advises against 4-bit on Qwen3.5
LORA = dict(r=32, lora_alpha=64, lora_dropout=0, use_gradient_checkpointing='unsloth', random_state=3407,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
!nvidia-smi -L

In [ ]:
import gc, json, torch, hl_lib
from unsloth import FastLanguageModel
N_MCQ = 300                                     # questions per task (500 available). Lower if time is short.
held = hl_lib.load_jsonl('ppl_heldout.jsonl'); mcq_all = hl_lib.load_jsonl('mcq_eval.jsonl'); vign = hl_lib.load_jsonl('vignettes.jsonl')
sources = sorted({d['source'] for d in held})
mcq = [it for t in ('MedQA', 'MedMCQA', 'PubMedQA') for it in [x for x in mcq_all if x['task'] == t][:N_MCQ]]
print({s: sum(d['source'] == s for d in held) for s in sources}, '| mcq items:', len(mcq), '| vignettes:', len(vign))

In [ ]:
results = json.load(open(f'{BASE}/eval_results.json')) if os.path.exists(f'{BASE}/eval_results.json') else {}
for name, path in [('base', MODEL), ('+cpt', f'{BASE}/adapter_cpt'), ('+cpt+sft', f'{BASE}/adapter_sft')]:
    if name in results: continue                # resume after a disconnect
    model, tok = FastLanguageModel.from_pretrained(path, max_seq_length=3072, **LOAD); FastLanguageModel.for_inference(model)
    r = {'ppl': {s: hl_lib.perplexity(model, tok, [d['text'] for d in held if d['source'] == s]) for s in sources}}
    r['mcq_plain'] = hl_lib.mcq_accuracy(model, tok, mcq, 'plain'); r['mcq_chat'] = hl_lib.mcq_accuracy(model, tok, mcq, 'chat')
    r['gen'] = [hl_lib.generate(model, tok, v['prompt'], mode='chat' if name == '+cpt+sft' else 'plain') for v in vign[:12]]
    results[name] = r; json.dump(results, open(f'{BASE}/eval_results.json', 'w'), indent=1)
    print(name, r['ppl'], r['mcq_plain'], r['mcq_chat'])
    del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import matplotlib.pyplot as plt, numpy as np
names = list(results); w = 0.25
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for i, n in enumerate(names): ax[0].bar(np.arange(len(sources)) + i * w, [results[n]['ppl'][s] for s in sources], w, label=n)
ax[0].set_xticks(np.arange(len(sources)) + w); ax[0].set_xticklabels(sources); ax[0].set_ylabel('perplexity (lower is better)'); ax[0].set_title('Held-out perplexity'); ax[0].legend()
tasks = ['MedQA', 'MedMCQA', 'PubMedQA']
for i, n in enumerate(names): ax[1].bar(np.arange(3) + i * w, [max(results[n]['mcq_plain'][t], results[n]['mcq_chat'][t]) for t in tasks], w, label=n)
ax[1].set_xticks(np.arange(3) + w); ax[1].set_xticklabels(tasks); ax[1].set_ylabel('accuracy (best of plain/chat prompt)'); ax[1].set_title('Zero-shot MCQ accuracy'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{BASE}/eval_summary.png', dpi=150); plt.show()
print('| model | ' + ' | '.join(f'PPL {s}' for s in sources) + ' | ' + ' | '.join(tasks) + ' |')
for n in names: print(f"| {n} | " + ' | '.join(f"{results[n]['ppl'][s]:.2f}" for s in sources) + ' | ' + ' | '.join(f"{max(results[n]['mcq_plain'][t], results[n]['mcq_chat'][t]):.3f}" for t in tasks) + ' |')

## GP vignettes side by side
Read these yourself. Do the answers name plausible conditions, red-flag symptoms, and sensible next steps? Note where the base model rambles and where the tuned one is structured. Chance is 25% for MedQA/MedMCQA (4 options) and about 33% for PubMedQA (the majority class 'yes' is about 55%).

In [ ]:
for i, v in enumerate(vign[:12]):
    print('=' * 100); print('PATIENT:', v['prompt'])
    for n in names: print(f'--- {n}:', results[n]['gen'][i][:600].replace('\n', ' '))